# 三种RAG模式复现查询菜谱系统

三种RAG
1. 基础rag ： 问，检索知识，把结果给llm，生成回答
2. agentic rag ： 问，agent判断是否调用tools，调用tools，回答 （检索变成Agent工具）
3. corrective rag ： 问，检索知识库，判断是否相关，相关回答，不相关则不回答

In [1]:
%pip install langchain langchain-openai langchain-community langchain-oceanbase
%pip install langchain-text-splitters pypdf pymysql python-dotenv
%pip install sentence-transformers langchain-huggingface pyobvector

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 23.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 683.6/683.6 kB 43.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 80.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 54.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 88.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.1/142.1 MB 30.7 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17/17 [langchain-oceanbase]kenizers]-hub]
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 2.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 20.0 MB/s  0:00:00eta 0:00:01
   

# 大致流程
```
准备环境
-> 启动 OceanBase
-> 加载文档
-> 文档切块
-> 向量化
-> 写入 OceanBase
-> 基础 RAG 问答
-> Agentic RAG
-> Corrective RAG：文档评分 + 托底
```

# 启动OceanBase数据库

In [ ]:
# 命令启动 ： docker compose -f docker-compose.oceanbase.yml up -d
# 检查docker状态 ： docker ps

# 配置.emv文件

配置文件如下
```.env
# 阿里百炼
BASE_URL=
API_KEY=

AIHUBMIX_API_KEY = 
AIHUBMIX_BASE_URL = 
AIHUBMIX_MODEL = "gpt-4.1-free"


#FAISS 向量数据库配置
FAISS_DB_DIR=./faiss_db

OB_HOST=127.0.0.1
OB_PORT=2881
OB_USER=root@test
OB_PASSWORD=
OB_DATABASE=test
```

# 基础RAG复现

In [15]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_oceanbase.vectorstores import OceanbaseVectorStore
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os 
import pymysql
from typing import List
from pathlib import Path

load_dotenv() # 加载环境变量

True

In [5]:
# 从环境变量中获取信息
args = {
    "ob_host": os.getenv("OB_HOST"),
    "ob_port": os.getenv("OB_PORT"),
    "ob_user": os.getenv("OB_USER"),
    "ob_password": os.getenv("OB_PASSWORD"),
    "ob_database": os.getenv("OB_DATABASE")
}

conn = pymysql.connect(
    host = args["ob_host"],
    port = int(args["ob_port"]),
    user = args["ob_user"],
    password = args["ob_password"],
    database = args["ob_database"],
    charset = "utf8mb4",
    connect_timeout = 10
)

# 测试链接
with conn.cursor() as cursor:
    cursor.execute("SELECT VERSION()")
    version = cursor.fetchone()
    print("OceanBase Version:", version[0])
    conn.close()

/opt/homebrew/Caskroom/miniconda/base/envs/pytorch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OceanBase Version: 5.7.25-OceanBase_CE-v4.3.3.1


In [ ]:
# 加载文档, 加切块, 加向量化

# 加载文档
def load_documents(data_path : str) -> List[Document] :
    """加载文档 : 输入是数据的路径, 然后md 和 text 还有 pdf 格式都会进行处理"""
    root = Path(data_path)
    docs : List[Document] = []
    for path in root.rglob("*"):
        if not path.is_file():
            continue

        suffix = path.suffix.lower() # 取这些路径的文件名
        if suffix in [".md", ".txt"]:
            text = path.read_text(encoding="utf-8", errors="ignore")
            docs.append(
                Document(
                    page_content=text, 
                    metadata={"source": str(path), "file_name": path.name},
                    )
            )
        elif suffix in [".pdf"]:
            docs.extend(PyPDFLoader(str(path)).load())

    return docs

# 进行分块
def split_documents(documents: List[Document]) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=100,
        separators=["\n\n", "\n", "。", ". ", " ", ""],
    )
    chunks = splitter.split_documents(documents)
    print(f"Split chunks: {len(chunks)}")
    return chunks

# 创建向量化模型
def get_embeddings_model() -> HuggingFaceEmbeddings:
    model_name = os.getenv("EMBEDDING_MODEL", "BAAI/bge-small-zh-v1.5")
    device = os.getenv("EMBEDDING_DEVICE", "cpu")
    return HuggingFaceEmbeddings(
        model_name = model_name,
        model_kwargs = {"device" : device},
        encode_kwargs = {"normalize_embeddings": True},
    )

In [12]:
# 创建向量数据库

def build_vector_store(chunks: List[Document], drop_old: bool) -> OceanbaseVectorStore:
    vectore_store = OceanbaseVectorStore(
        embedding_function=get_embeddings_model(),
        table_name=os.getenv("OB_TABLE_NAME", "langchain_knowledge_base"),
        connection_args=args,
        vidx_metric_type="l2", # 计算相似度的方式使用l2
        index_type="HNSW",
        drop_old=drop_old,
        normalize=True,
    )

    if chunks :
        ids = vectore_store.add_documents(chunks)
        print(f"Add documents : {len(ids)}")

In [14]:
# 创建llm
def get_llm(temperature : float = 0) -> ChatOpenAI:
    api_key = os.getenv("AIHUBMIX_API_KEY")
    base_url = os.getenv("AIHUBMIX_BASE_URL")
    model = os.getenv("AIHUBMIX_MODEL", "gpt-4o-mini")

    if not api_key:
        raise ValueError("请设置ai hub api key")
    
    return ChatOpenAI(
        model=model,
        temperature=temperature,
        api_key=api_key,
        base_url=base_url
    )

In [16]:
# 格式化docs,给他添加上必要的信息
def format_docs(docs: List[Document], max_chars: int = 3500) -> str:
    parts = []
    total = 0
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source", "unknown")
        text = f"[Document {i}] source={source}\n{doc.page_content}\n"
        if total + len(text) > max_chars:
            break
        parts.append(text)
        total += len(text)
    return "\n".join(parts)

def generate_answer(question : str, docs : List[Document]) -> str:
    llm = get_llm()
    context = format_docs(docs)

    prompt = ChatPromptTemplate(
        """你是一个严谨的 RAG 问答助手。
        请只根据给定上下文回答用户问题。
        如果上下文不足，请直接说“资料不足，无法根据知识库回答”。

        用户问题：
        {question}

        上下文：
        {context}

        回答："""
    )

    chain = prompt | llm 

    # 这里传入的信息取决于这个模版prompt,  所以不是调用llm时候传入的user 和 message信息
    result = chain.invoke({"question" : question, "context" : context})

    return result.content



In [18]:
def grade_documents(question: str, docs: List[Document]) -> dict:
    llm = get_llm(temperature=0)
    context = format_docs(docs)

    # 下面双重大括号代表一组括号, 不是填空
    prompt = ChatPromptTemplate.from_template(
        """你是一个文档相关性评分器。
        请判断“检索到的文档”是否能帮助回答“用户问题”。

        只返回 JSON，不要返回多余解释：
        {{"relevant": true 或 false, "reason": "简短理由"}}

        用户问题：
        {question}

        检索到的文档：
        {context}
        """
    )
    result = (prompt | llm).invoke(
        {"question": question, "context": context}
    )

    import json
    import re

    text = result.content.strip()
    match = re.search(r"\{.*\}", text, flags=re.S) # 这里只匹配大括号
    if not match:
        return {"relevant": False, "reason": f"评分器没有返回 JSON：{text[:100]}"}

    try:
        data = json.loads(match.group(0)) # 把字符串转化成json 字典
        return {
            "relevant": bool(data.get("relevant")),
            "reason": str(data.get("reason", "")),
        }
    except Exception as exc:
        return {"relevant": False, "reason": f"JSON 解析失败：{exc}"}

In [19]:
def fallback_answer(question: str) -> str:
    return (
        "当前知识库检索结果与问题不相关，已触发托底。\n\n"
        f"你的问题是：{question}\n\n"
        "新手版暂时不接入真实 Web Search，因此这里先返回安全提示："
        "请补充相关资料到知识库，或者接入 Tavily/Bing Search 作为外部搜索工具。"
    )